# Teste isolado — ARTEMIG (Notícias)

Fonte candidata: **ARTEMIG - Agência Reguladora de Serviços Públicos
Delegados de Minas Gerais**, setor Transporte. Já existia placeholder em
`controle_fontes` (`source_id='—'`, `status='Não iniciada'`,
`importancia_original='Média'`). Notebook **descartável** (Fase 1) — sem
dispatcher, sem `atualizar_status_fonte`, sem gravar nada em produção.

URL fornecida: `https://artemig.mg.gov.br/`.

## Nota operacional — certificado TLS incompleto

O servidor devolve uma cadeia de certificado incompleta ("unable to get
local issuer certificate" -- falta o intermediário) para clientes que
validam contra o bundle padrão do SO. `httpx`/`requests` no Databricks
costumam usar o bundle do `certifi`, que às vezes já cobre o
intermediário faltante -- testar sem `verify=False` primeiro; só usar
`verify=False` como último recurso, e documentar se for necessário.

In [ ]:
%pip install --quiet httpx beautifulsoup4 lxml
dbutils.library.restartPython()

In [ ]:
import httpx
from bs4 import BeautifulSoup

BASE = "https://artemig.mg.gov.br"
USER_AGENT = (
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/120.0 Safari/537.36"
)
HEADERS = {"User-Agent": USER_AGENT, "Accept-Language": "pt-BR,pt;q=0.9"}

## Teste 1 — robots.txt e a home

WordPress + Yoast SEO. `robots.txt` só bloqueia busca interna e
`/wp-json/` -- não impede scraping de conteúdo. A home tem uma seção
"Últimas notícias" com botão "Ver todas as notícias" apontando para
`/category/noticias/`.

In [ ]:
with httpx.Client(headers=HEADERS, timeout=30, follow_redirects=True) as client:
    resp_robots = client.get(f"{BASE}/robots.txt")
    resp_home = client.get(f"{BASE}/")

print(f"/robots.txt -> HTTP {resp_robots.status_code}\n{resp_robots.text}\n")
print(f"/ -> HTTP {resp_home.status_code}, {len(resp_home.text)} chars")

soup_home = BeautifulSoup(resp_home.text, "lxml")
titulo_secao = soup_home.find(string=lambda t: t and "Últimas notícias" in t)
print(f"Seção 'Últimas notícias' encontrada na home: {titulo_secao is not None}")

## Teste 2 — `/category/noticias/` (linkado pela própria home) está vazio

**Achado**: a categoria "Notícias" -- o próprio link "Ver todas as
notícias" da home aponta pra cá -- devolve "Nenhum post encontrado".
Bug/lacuna de conteúdo do lado da ARTEMIG, não nosso.

In [ ]:
with httpx.Client(headers=HEADERS, timeout=30, follow_redirects=True) as client:
    resp_cat = client.get(f"{BASE}/category/noticias/")

print(f"HTTP {resp_cat.status_code}, {len(resp_cat.text)} chars")
print(f"'Nenhum post encontrado' presente: {'Nenhum post encontrado' in resp_cat.text}")

## Teste 3 — `/posts/` (índice padrão do WordPress) tem posts, mas são
## páginas institucionais, não notícias

`post-sitemap.xml` (Yoast) mostrava atividade recente (última mod.
13/08/2026), então existe algum fluxo de publicação ativo. O índice de
posts de fato (`/posts/`, não a categoria vazia) lista itens reais --
mas verificando os títulos das duas primeiras páginas (20 itens, cobrindo
~1 ano, agosto/2025 a junho/2026), **nenhum é notícia**: são todos
páginas institucionais publicadas como `post` do WordPress (Plano de
Comunicação, Editais, Acessibilidade, LGPD, cada trecho de rodovia
concedida, etc.) -- o mesmo conteúdo que já aparece no menu institucional
do site, sem nenhum evento/decisão/anúncio datado.

In [ ]:
import re

titulos_todos = []
with httpx.Client(headers=HEADERS, timeout=30, follow_redirects=True) as client:
    for pagina in [1, 2]:
        url = f"{BASE}/posts/" if pagina == 1 else f"{BASE}/posts/page/{pagina}/"
        resp = client.get(url)
        titulos = re.findall(r'<h2 class="entry-title">\s*<a href="([^"]*)">([^<]*)</a>', resp.text)
        titulos_todos.extend(titulos)

print(f"{len(titulos_todos)} posts encontrados em /posts/ (2 páginas):\n")
for url, titulo in titulos_todos:
    print(f"  {titulo.strip()}")

## Teste 4 — achado: existe conteúdo regulatório real, só não é "notícia"

`category-sitemap.xml` (Yoast) lista 7 categorias, incluindo `portarias`
e `resolucoes_conjuntas` com `lastmod` recente (07/08/2026) -- mas
`/category/portarias/` e `/category/resolucoes_conjuntas/` também
devolvem "Nenhum post encontrado" (mesmo bug de arquivo de categoria já
visto em `/category/noticias/`, aparentemente **sitewide**, não
específico de notícias).

O conteúdo real está em outro lugar: `dtic_documento-sitemap.xml` revela
um **custom post type** (`dtic_documento`) com ~42 documentos
(Portarias, Resoluções, Resoluções Conjuntas, Leis), atualizado até
05-07/08/2026 -- ativo de verdade. A listagem funcional (que não sofre
do bug de categoria) é `/documentos/` (índice do custom post type, 5
páginas, ~10 itens/página).

In [ ]:
with httpx.Client(headers=HEADERS, timeout=30, follow_redirects=True) as client:
    resp_cat_portarias = client.get(f"{BASE}/category/portarias/")
    resp_documentos = client.get(f"{BASE}/documentos/")

print(f"/category/portarias/ -> 'Nenhum post encontrado': "
      f"{'Nenhum post encontrado' in resp_cat_portarias.text} (mesmo bug sitewide)")
print(f"/documentos/ -> HTTP {resp_documentos.status_code}, {len(resp_documentos.text)} chars")

soup_documentos = BeautifulSoup(resp_documentos.text, "lxml")
cards = soup_documentos.select("article.post-card")
print(f"{len(cards)} documentos na página 1 de /documentos/")

## Teste 5 — listar documentos (título, data, link da página) e navegar
## até 5 páginas

Título/data prontos em cada card da listagem (mesmo padrão de
`/posts/`), mas o link de cada card aponta pra **página HTML do
documento**, não direto pro PDF -- precisa de uma segunda requisição
por item.

In [ ]:
import re

MESES_PT_ARTEMIG = {
    "janeiro": 1, "fevereiro": 2, "março": 3, "marco": 3, "abril": 4,
    "maio": 5, "junho": 6, "julho": 7, "agosto": 8, "setembro": 9,
    "outubro": 10, "novembro": 11, "dezembro": 12,
}

def listar_pagina_documentos(html: str) -> list[dict]:
    soup = BeautifulSoup(html, "lxml")
    itens = []
    for card in soup.select("article.post-card"):
        tag_a = card.select_one("h2.entry-title a[href]")
        if not tag_a:
            continue
        titulo = tag_a.get_text(" ", strip=True)
        url_doc = tag_a["href"].strip()

        published_at = None
        tag_data = card.select_one(".posted-on")
        if tag_data:
            m = re.search(r"(\d{1,2}) de (\w+) de (\d{4})", tag_data.get_text(strip=True))
            if m:
                dia, mes_nome, ano = m.groups()
                mes = MESES_PT_ARTEMIG.get(mes_nome.lower())
                if mes:
                    published_at = f"{ano}-{mes:02d}-{int(dia):02d}"

        itens.append({"titulo": titulo, "url": url_doc, "published_at": published_at})
    return itens


todos_documentos = []
with httpx.Client(headers=HEADERS, timeout=30, follow_redirects=True) as client:
    for pagina in range(1, 6):
        url = f"{BASE}/documentos/" if pagina == 1 else f"{BASE}/documentos/page/{pagina}/"
        resp = client.get(url)
        itens_pagina = listar_pagina_documentos(resp.text)
        if not itens_pagina:
            break
        todos_documentos.extend(itens_pagina)

print(f"{len(todos_documentos)} documentos no total (todas as páginas).\n")
for item in todos_documentos[:5]:
    print(f"{item['published_at'] or '?':<12} {item['titulo'][:80]}")

## Teste 6 — abrir uma página de documento e extrair o PDF real

Cada página de documento tem um ou mais links `a.dtic-documento-file`.
**Cuidado**: nem todos apontam pra `.pdf` -- documentos do tipo "Lei"
linkam pro portal externo da ALMG (`almg.gov.br/legislacao-mineira/...`),
uma página HTML, não um arquivo. Preciso filtrar só links que terminam
em `.pdf`, e pular o item (contar como "sem PDF direto") se nenhum
link do documento for PDF de verdade -- mesmo padrão já usado pra
ARPE.

In [ ]:
def extrair_pdf_documento(html: str) -> str | None:
    soup = BeautifulSoup(html, "lxml")
    for tag_a in soup.select("a.dtic-documento-file[href]"):
        href = tag_a["href"].strip()
        if href.lower().endswith(".pdf"):
            return href
    return None


amostras = todos_documentos[:3] + [item for item in todos_documentos if "lei" in item["titulo"].lower()][:1]

with httpx.Client(headers=HEADERS, timeout=30, follow_redirects=True) as client:
    for item in amostras:
        resp_doc = client.get(item["url"])
        url_pdf = extrair_pdf_documento(resp_doc.text)
        print(f"{item['titulo'][:70]:<70} -> {url_pdf or 'SEM PDF DIRETO (ex.: Lei -> portal ALMG)'}")

## Conclusão da Fase 1 (revisada — investigação completa)

**Notícias**: confirmado que não há nenhuma publicada (Teste 1-3, ver
acima) -- widget da home vazio, categoria vazia, `/posts/` só tem
páginas institucionais.

**Mas existe conteúdo regulatório real e ativo**: custom post type
`dtic_documento` (Portarias, Resoluções, Resoluções Conjuntas, Leis) com
~42 documentos, atualizado até 07/08/2026 -- mesmo padrão de PDFs
regulatórios já coberto pra ARPE/AGESAN-RS/CCEE. Listagem funcional em
`/documentos/` (5 páginas), com segunda requisição por item pra achar o
PDF real na página do documento (nem todo item tem PDF -- "Lei" aponta
pro portal externo da ALMG).

**Avaliação para a Fase 2**: encaixa no dispatcher genérico `ingest-PDF`
(mesmo grupo de ARPE/AGESAN/CCEE) -- "baixar uma URL e extrair PDFs de
forma padrão", com um `listar_artemig_documentos()` próprio (2 níveis de
requisição: listagem paginada + página de cada documento pra achar o
`.pdf`, filtrando por extensão pra pular casos como "Lei"). Histórico
pequeno (~42 documentos) -- `max_paginas=5` cobre tudo numa primeira
execução.